In [ ]:
# Install all required packages
!pip install git+https://github.com/openai/CLIP.git
!pip install -q -U llama-index chromadb llama-index-vector-stores-chroma llama-index-embeddings-clip llama-index-readers-json

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-n7l0px8f
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-n7l0px8f
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=876d42eb1b19a7a69302f8fbb3e21ef40cf4a487c6a01f5a57969518f93a3062
  Stored in directory: /tmp/pip-ephem-wheel-cache-v37nmatu/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 97.6 MB/s eta 0:00:00


In [ ]:
import chromadb, json
from typing import List, Dict

# -- LLama Index Imports -- #
from llama_index.embeddings.clip import ClipEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.indices import MultiModalVectorStoreIndex
from llama_index.core import StorageContext, SimpleDirectoryReader, Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.node_parser import SimpleFileNodeParser

In [ ]:
# --- Set up the Index and Retriever --- #
db = chromadb.PersistentClient(path="multimodal_db")
text_collection = db.get_or_create_collection("text_collection")
image_collection = db.get_or_create_collection("image_collection")
text_store = ChromaVectorStore(chroma_collection=text_collection)
image_store = ChromaVectorStore(chroma_collection=image_collection)
storage_context = StorageContext.from_defaults(vector_store=text_store, image_store=image_store)
clip_embed = ClipEmbedding()

# --- JSON Text Handling --- #

# --- APOD Summaries ---#
# Custom parsing function
def parse_apod_explanations(data: List[Dict]) -> List[str]:
    """
    Parses a list of APOD entries and extracts only the 'explanation' text.
    """
    explanations = []
    for entry in data:
        if "explanation" in entry:
            explanations.append(entry["explanation"])
    return explanations

# Open and load the JSON file using Python's built-in library
file_path = "/content/drive/MyDrive/APOD/DATA/TEXT/apod_data.json"
with open(file_path, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

# Use function to process the loaded data into a list of strings
extracted_texts = parse_apod_explanations(json_data)

# Manually create LlamaIndex Document objects from text
apod_documents = [Document(text=t) for t in extracted_texts]

print(f"APOD Documents: {apod_documents[5]}")

# --- OpenStax Textbook --- #
# Define the parsing function for the OpenStax data structure
def parse_openstax_values(data: Dict[str, str]) -> List[str]:
    """
    Parses an OpenStax-style dictionary and extracts all text values.
    """
    # .values() gets all the text snippets, and list() converts them
    return list(data.values())

# Open and load OpenStax JSON file
file_path = "/content/drive/MyDrive/APOD/DATA/TEXT/OpenStax_Astronomy2e.json"
with open(file_path, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

# Use the function to extract the text
extracted_texts = parse_openstax_values(json_data)

# Manually create LlamaIndex Document objects
os_documents = [Document(text=t) for t in extracted_texts]

print(f"OpenStax Documents: {os_documents[5]}")

text_docs = apod_documents + os_documents

# Exclude metadata from the embedding
for d in text_docs:
    d.excluded_embed_metadata_keys.extend([
        "file_name",
        "file_type",
        "file_size",
        "creation_date",
        "last_modified_date",
        "file_path"
    ])

splitter = SentenceSplitter(chunk_size=45, chunk_overlap=10)

# Split all documents into nodes
text_nodes = []
for doc in text_docs:
    text_nodes.extend(splitter.get_nodes_from_documents([doc]))

# --- Image Handling --- #
image_docs = SimpleDirectoryReader("/content/drive/MyDrive/APOD/DATA/IMAGES").load_data()
parser = SimpleFileNodeParser()
image_nodes = parser.get_nodes_from_documents(image_docs)

# --- Index Creation --- #
index = MultiModalVectorStoreIndex(
    [],
    storage_context=storage_context,
    embed_model=clip_embed,
    image_embed_model=clip_embed,
    show_progress = True,
)

BATCH_SIZE = 5461  # Must be <= vector store's max batch size

# Insert image nodes in batches
for i in range(0, len(image_nodes), BATCH_SIZE):
  batch_nodes = image_nodes[i:i+BATCH_SIZE]
  index.insert_nodes(batch_nodes)
  print(f"Inserted {i+BATCH_SIZE} images")

# Insert text nodes in batches
for i in range(0, len(text_nodes), BATCH_SIZE):
    batch_nodes = text_nodes[i:i+BATCH_SIZE]
    index.insert_nodes(batch_nodes)
    print(f"Inserted {i+BATCH_SIZE} texts")

100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 214MiB/s]


Streaming output truncated to the last 5000 lines.
Metadata length (0) is close to chunk size (45). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
Metadata length (0) is close to chunk size (45). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
Metadata length (0) is close to chunk size (45). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
Metadata length (0) is close to chunk size (45). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
Metadata length (0) is close to chunk size (45). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
Metadata length (0) is close to chunk size (

In [ ]:
# Node type testing
image_nodes = index.image_vector_store._get(limit=70993, where={}).nodes
print(f"Total image nodes in Chroma: {len(image_nodes)}")
for node in image_nodes[:10]:
    print("Node ID:", node.node_id)
    print("Type:", type(node))
    print("Node Content:", node.get_content() + "...")
    print("Metadata:", node.metadata)
    print("-" * 40)

Total image nodes in Chroma: 70262
Node ID: 36706c55-fc1f-4700-a3d9-847831c96c2a
Type: <class 'llama_index.core.schema.TextNode'>
Node Content: Today's Picture:    Explanation:  If the Earth could somehow be transformed to the ultra-high density of a neutron star , it might appear as it does in the above computer generated figure....
Metadata: {}
----------------------------------------
Node ID: 9d579fa9-1e27-44aa-9295-4212bc6614ab
Type: <class 'llama_index.core.schema.TextNode'>
Node Content: Due to the very strong gravitational field, the neutron star distorts light from the background sky greatly. If you look closely, two images of the constellation Orion are visible....
Metadata: {}
----------------------------------------
Node ID: fd68d056-6d58-4313-898e-09395fbdf134
Type: <class 'llama_index.core.schema.TextNode'>
Node Content: The gravity of this particular neutron star is so great that no part of the neutron star is blocked from view - light is pulled around by gravity even fro

In [ ]:
# Save Index and DB
index.storage_context.persist(persist_dir="/content/index")
!cp -r /content/index /content/drive/MyDrive/APOD
!cp -r /content/multimodal_db /content/drive/MyDrive/APOD